In [1]:
import pandas as pd
import requests,io

### 避開SSL驗證


In [2]:
#api_url="https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=af57253c-e838-46da-a1f5-12b43afd75f3&limit=1000&sort=datacreationdate%20desc&format=CSV"
api_url="https://data.moenv.gov.tw/api/v2/aqx_p_02?api_key=846e44e1-8cc5-4893-ad87-c79d2d383706&limit=1000&sort=datacreationdate%20desc&format=JSON"
resp=requests.get(api_url,verify=False)
df=pd.read_json(io.StringIO(resp.text))
df

c:\Users\User\OneDrive\桌面\Python Web開發(Django)\mysql\pm25-opendata\.venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'data.moenv.gov.tw'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,16,2026-04-26 12:00,μg/m3
1,員林,彰化縣,14,2026-04-26 12:00,μg/m3
2,臺灣大道,臺中市,11,2026-04-26 12:00,μg/m3
3,大城,彰化縣,10,2026-04-26 12:00,μg/m3
4,富貴角,新北市,13,2026-04-26 12:00,μg/m3
...,...,...,...,...,...
995,臺南,臺南市,22,2026-04-26 00:00,μg/m3
996,安南,臺南市,19,2026-04-26 00:00,μg/m3
997,善化,臺南市,28,2026-04-26 00:00,μg/m3
998,新營,臺南市,18,2026-04-26 00:00,μg/m3


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   site              1000 non-null   str  
 1   county            1000 non-null   str  
 2   pm25              1000 non-null   str  
 3   datacreationdate  1000 non-null   str  
 4   itemunit          1000 non-null   str  
dtypes: str(5)
memory usage: 39.2 KB


In [4]:
df.describe()

,site,county,pm25,datacreationdate,itemunit
count,1000,1000,1000,1000,1000
unique,80,22,42,13,1
top,林森,高雄市,11,2026-04-26 12:00,μg/m3
freq,13,156,77,80,1000


### 清理資料(去空值)跟去重複

In [5]:
df.head(5)

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,16,2026-04-26 12:00,μg/m3
1,員林,彰化縣,14,2026-04-26 12:00,μg/m3
2,臺灣大道,臺中市,11,2026-04-26 12:00,μg/m3
3,大城,彰化縣,10,2026-04-26 12:00,μg/m3
4,富貴角,新北市,13,2026-04-26 12:00,μg/m3


### 確認重複資料

In [6]:
df2=df[df.duplicated(subset=["site","datacreationdate"])]
df2

,site,county,pm25,datacreationdate,itemunit


### subset 約定重複跟刪除nan的依據

In [7]:
df1=df.drop_duplicates(subset=["site","datacreationdate"]).dropna()
df1

,site,county,pm25,datacreationdate,itemunit
0,林森,臺南市,16,2026-04-26 12:00,μg/m3
1,員林,彰化縣,14,2026-04-26 12:00,μg/m3
2,臺灣大道,臺中市,11,2026-04-26 12:00,μg/m3
3,大城,彰化縣,10,2026-04-26 12:00,μg/m3
4,富貴角,新北市,13,2026-04-26 12:00,μg/m3
...,...,...,...,...,...
995,臺南,臺南市,22,2026-04-26 00:00,μg/m3
996,安南,臺南市,19,2026-04-26 00:00,μg/m3
997,善化,臺南市,28,2026-04-26 00:00,μg/m3
998,新營,臺南市,18,2026-04-26 00:00,μg/m3


### sqlite 建立資料庫

In [8]:
import sqlite3

In [9]:
# unique  插入資料唯一的約束
sqlstr='''
create table if not exists data(
id integer primary key autoincrement,
site text,
county text,
pm25 integer,
datacreationdate text,
itemunit text,
unique(site,datacreationdate)
)
'''

In [10]:
conn=sqlite3.connect("pm25.db")
cursor=conn.cursor()

conn,cursor

(<sqlite3.Connection at 0x1237fe22d40>, <sqlite3.Cursor at 0x1237e896940>)

In [11]:
cursor.execute(sqlstr)
conn.commit()


In [12]:
sqlstr='insert or ignore into data (site,county,pm25,datacreationdate,itemunit)\
    values(?,?,?,?,?)'

In [13]:
df1.values.tolist()

[['林森', '臺南市', '16', '2026-04-26 12:00', 'μg/m3'],
 ['員林', '彰化縣', '14', '2026-04-26 12:00', 'μg/m3'],
 ['臺灣大道', '臺中市', '11', '2026-04-26 12:00', 'μg/m3'],
 ['大城', '彰化縣', '10', '2026-04-26 12:00', 'μg/m3'],
 ['富貴角', '新北市', '13', '2026-04-26 12:00', 'μg/m3'],
 ['麥寮', '雲林縣', '9', '2026-04-26 12:00', 'μg/m3'],
 ['關山', '臺東縣', '2', '2026-04-26 12:00', 'μg/m3'],
 ['馬公', '澎湖縣', '19', '2026-04-26 12:00', 'μg/m3'],
 ['金門', '金門縣', '17', '2026-04-26 12:00', 'μg/m3'],
 ['馬祖', '連江縣', '21', '2026-04-26 12:00', 'μg/m3'],
 ['埔里', '南投縣', '11', '2026-04-26 12:00', 'μg/m3'],
 ['復興', '高雄市', '16', '2026-04-26 12:00', 'μg/m3'],
 ['永和', '新北市', '17', '2026-04-26 12:00', 'μg/m3'],
 ['竹山', '南投縣', '16', '2026-04-26 12:00', 'μg/m3'],
 ['中壢', '桃園市', '16', '2026-04-26 12:00', 'μg/m3'],
 ['三重', '新北市', '15', '2026-04-26 12:00', 'μg/m3'],
 ['冬山', '宜蘭縣', '11', '2026-04-26 12:00', 'μg/m3'],
 ['宜蘭', '宜蘭縣', '11', '2026-04-26 12:00', 'μg/m3'],
 ['陽明', '臺北市', '13', '2026-04-26 12:00', 'μg/m3'],
 ['花蓮', '花蓮縣', '13', '2026-04-

In [14]:
cursor.executemany(sqlstr,df1.values.tolist())
conn.commit()

In [15]:
cursor.rowcount

240